# EDA — Reconciliation-Focused Diagnostics

Every plot defends a specific modeling decision.  All computation imports from
the module layer (`data.py`, `covariance.py`, `eda.py`); no logic is re-implemented here.

**Decision map:**
| Figure | Decision defended |
|--------|-------------------|
| 1 — Incoherence over time | Motivates the task: base preds violate hierarchy identities |
| 2 — Heteroscedasticity | Scaled (relative) residuals + per-day W_day rescaling |
| 3 — Volume trend | Per-day rescaling; a single absolute W would be dominated by peak days |
| 4 — Bias | Disclose-don't-fix: reconciliation cannot address this |
| 5 — ACF | OOS W estimation; SS lambda underestimates warranted shrinkage |
| 6 — W structure | Reliability ranking: who gets moved and by how much |
| 7 — Sibling correlations | Direction of adjustments: who pulls whom |
| 8 — Residual tails | Covariance-based MinT is a reasonable approximation but not exact |
| 9 — Rolling stability | Full-train C is stable enough for the future period |

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from data import SERIES, S, load, pred_array, actual_array
from covariance import relative_residuals, schafer_strimmer, effective_n_sensitivity
import eda

FIGURES_DIR = 'figures'
import os; os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
train, future, gap_info = load()
print('Train rows:', gap_info['n_train'], '  Future rows:', gap_info['n_future'])
print('Max date gap in train:', gap_info['max_gap_days'], 'day(s)')
if gap_info['gap_dates']:
    print('Gap at:', [str(d.date()) for d in gap_info['gap_dates']])

# Fit relative covariance once on all train data
X_rel = relative_residuals(train, SERIES)
C, lam = schafer_strimmer(X_rel)
eff = effective_n_sensitivity(X_rel, lam, len(train))
print(f'\nSS lambda: {lam:.5f}')
print(f'n_eff (cross-product lag-1 rho={eff["rho_cross_product"]:.3f}): {eff["n_eff_cross_product"]:.0f}')
print(f'lambda inflated (cross-product): {eff["lambda_inflated_cp"]:.5f}')
print(f'n_eff (level-residual proxy, rho={eff["rho_level_residual"]:.3f}): {eff["n_eff_level_residual"]:.0f}')
print(f'lambda inflated (level-residual proxy): {eff["lambda_inflated_lv"]:.5f}')

## Figure 1: Base-Prediction Incoherence Over Time

**Decision defended:** The entire reconciliation task is justified here.  
Train actuals are perfectly coherent (floating-point noise only, gaps < 4e-9).  
Base predictions violate hierarchy identities: train gap mean ~13k / max ~463k;  
future gap mean ~2.2M / max ~56M (the stated max of ~5.6e7 matches the future, not train).  
The future incoherence is ~170× larger than the train incoherence, consistent with the  
~10× volume growth and likely longer forecast horizon for future predictions.

In [ ]:
fig = eda.plot_incoherence(train, future, save_dir=FIGURES_DIR)
plt.show()

## Figure 2: Heteroscedasticity

**Decision defended:** Scaled (relative) residuals; per-day W_day = diag(pred) @ C @ diag(pred).  

Three panels:
- **A.** |residual| vs level scatter for aggregate.  Two correlation estimates are shown:
  - `corr(level=pred) = 0.62–0.73` — the operationally relevant metric (pred is available at forecast time)
  - `corr(level=actual) = 0.67–0.82` — higher because actual also embeds the residual and both track the day's scale.  In this data, actual = pred + residual, so actual carries part of the same variation; the ordering is an empirical tendency specific to this heteroscedastic data, not a mathematical guarantee (a simulation with symmetric independent residuals shows both correlations near zero).
- **B.** Residual std by volume quintile: top quintile (mean ~9M) has std ~2M (25% of level); lower quintiles 12–15%.
- **C.** Raw vs relative residual spread: raw residuals span orders of magnitude across series; relative residuals are more homoscedastic.

In [ ]:
fig = eda.plot_heteroscedasticity(train, save_dir=FIGURES_DIR)
plt.show()

## Figure 3: Volume Trend and Seasonality

**Decision defended:** Per-day W_day rescaling.  
Aggregate monthly mean grows ~10× over 3 years (first 3 months ~855k; last 3 months ~8.4M mean).  
A single absolute covariance W would be dominated by November peak-period days (~14M);  
relative residuals + per-day scaling ensure W reflects the error structure at each day's scale.  
November spike appears every year with roughly doubling YoY; April is also elevated.

In [ ]:
fig = eda.plot_volume_trend(train, save_dir=FIGURES_DIR)
plt.show()

## Figure 4: Forecast Bias

**Decision defended:** Disclose, do not attempt to fix via reconciliation.  

Mean residual (actual - pred) is positive for all 7 series: 1.1%–4.1% of mean pred level.  
However, the base model under-predicts on only 44.5%–48.9% of days — it over-predicts more  
than half the time.  The positive mean bias comes from a small number of large under-shoots  
on high-volume days (magnitude dominates frequency).  

Reconciliation redistributes within-day across the hierarchy; it cannot change the  
sign or magnitude of the per-day total forecast error.

In [ ]:
fig = eda.plot_bias(train, save_dir=FIGURES_DIR)
plt.show()

## Figure 5: Residual Autocorrelation

**Decision defended:** Out-of-sample (past-only) W estimation in the backtest;  
and noting that the SS analytic lambda is derived under IID assumptions.  

Lag-1 autocorr 0.34–0.56 across series; max same-sign error streak 23 days (cohort_B, B1).  
Errors are not iid in time — persistent over-prediction or under-prediction runs of 3+ weeks  
are common.  This has two implications:
1. Using IID W (estimated from shuffled residuals) would overstate effective sample size.
2. The SS lambda likely underestimates warranted shrinkage (inflated lambda still ~0.016–0.024, tiny).

In [ ]:
fig = eda.plot_acf(train, save_dir=FIGURES_DIR)
plt.show()

## Figure 6: Estimated W — Correlation Structure and Reliability

**Decision defended:** Full 7×7 W (not leaves-only); reliability-weighted reconciliation.  

Panel A shows strong positive correlations throughout (all series track the same volume driver).  
Panel B ranks series by relative variance (diagonal of C): higher = less reliable = moved more.  
Series with high relative variance are downweighted in the GLS objective, so reconciliation  
moves them toward the consensus of more reliable series.

In [ ]:
fig = eda.plot_W_structure(C, SERIES, save_dir=FIGURES_DIR)
plt.show()

## Figure 7 (Additional): Sibling and Cross-Cohort Correlations

**Why this matters for reconciliation:**  
The direction of adjustments is set by off-diagonal correlations in C.  
High sibling correlation (A1-A2, B1-B2) means when one sibling is pulled up/down,  
the other is pushed in the same direction.  Cross-cohort correlations set the  
direction of within-day reallocations between cohort_A and cohort_B.

In [ ]:
fig = eda.plot_sibling_correlations(C, SERIES, train, save_dir=FIGURES_DIR)
plt.show()

## Figure 8 (Additional): Residual Non-Gaussianity

**Why this matters:**  
MinT/GLS covariance weighting is optimal under Gaussian errors.  Heavy tails  
(excess kurtosis > 0) indicate outlier days where squared-loss covariance weighting  
may under-penalise large deviations.  This does not invalidate the method —  
covariance-based reconciliation is still consistent — but it implies the GLS weights  
are approximate under the true error distribution.  

**Log transform note:** Log-transforming before reconciliation would better handle  
multiplicative noise but breaks the additive coherence identities  
(aggregate ≠ cohort_A + cohort_B in log space by Jensen's inequality).  
Relative residuals achieve the same heteroscedasticity correction for covariance  
estimation while keeping the reconciliation in the original additive space.

In [ ]:
fig = eda.plot_residual_tails(train, save_dir=FIGURES_DIR)
plt.show()

## Figure 9 (Additional): Rolling Correlation Stability

**Decision defended:** Using full-train C for future reconciliation.  

If rolling correlations drift substantially over time, a recent-window estimator  
would be preferable over the full-train average.  If they are broadly stable,  
the full-train C is a safe choice with lower estimation variance.  

Note that the November spike in aggregate volume could cause seasonal shifts in  
the correlation structure during peak months.

In [ ]:
fig = eda.plot_rolling_stability(train, save_dir=FIGURES_DIR)
plt.show()